## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [4]:
import sys
!"{sys.executable}" -m pip install faiss-cpu sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
  Using cached faiss_cpu-1.13.2-cp313-cp313-win_amd64.whl.metadata (7.6 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.5.0-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
Using cached faiss_cpu-1.13.2-cp313-cp313-win_amd64.whl (18.9 MB)
   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ----------------------------------- ---- 524.3/588.7 kB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 588.7/588.7 kB 3.0 MB/s  0:00:00
   ---------------------------------------- 0.0/10.6 MB 


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\valer\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import faiss
print("faiss ok")

In [5]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

C:\Users\valer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [16]:
MY_BUBBLE_FILE = "personalist_salvator.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: personalist_salvator
Texte: 50


,id,agent,text
0,yt_X3bwh1-9nUU_Ugxc3Zx_bRhxFU18gXp4AaABAg,Personalist-salvator,"Ne au distrus hoții 😢,,,nu ne mai aparține nim..."
1,yt_bee6nXyzJ_E_UgyPoZoAIURM885eTr94AaABAg,Personalist-salvator,"Era si timpul sa treceti la atac, prea multa d..."
2,yt_bee6nXyzJ_E_UgzmsNNAxtBIZ34dyMB4AaABAg,Personalist-salvator,Vă mulțumim și noi și copii noștri care lucrea...
3,yt_bee6nXyzJ_E_UgzdyYPCNenomSY1CGN4AaABAg,Personalist-salvator,1:35 NOI știm că Sistemul ÎL hărțuiește mișele...
4,yt_bee6nXyzJ_E_UgwsuxSiNBqVBBSlPt14AaABAg,Personalist-salvator,Ideea că poporul are dreptul sau chiar datoria...


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [18]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Ne au distrus hoții 😢,,,nu ne mai aparține nimic,,,, și au pus labele spurcate pe o țară in 89,,,,,,cu datorie externă 0,,,,,,,.


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [19]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

Batches: 100%|██████████| 2/2 [00:03<00:00,  1.77s/it]

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [27]:
# Bula mea are 50 de texte.
# Au fost generați 50 de vectori.
# A doua valoare din embeddings.shape reprezintă dimensiunea fiecărui vector embedding, adică 384.

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [28]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\personalist_salvator
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [39]:
index_file = out_dir / "index.faiss"
metadata_file = out_dir / "index.pkl"

print("index.faiss există:", index_file.exists())
print("index.pkl există:", metadata_file.exists())
print("index.ntotal este egal cu numărul de texte:", index.ntotal == len(texts))

index.faiss există: True
index.pkl există: True
index.ntotal este egal cu numărul de texte: True


## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [48]:
# Text nou introdus în aplicație

input_text = "Poporul s-a săturat de politicienii controlați de sistem și are nevoie de un lider care să apere cu adevărat România."

In [49]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [50]:
# query_vector

In [51]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.665
Text: In sfarsit cineva care pune punctul pe I direct. Cei mai rai oameni i-am vazut in biserica! Romania a devenit din democratie o teocratie corupta.

Rezultat 2
Scor: 0.578
Text: Dragul meu Robert,eu sunt un nimeni si poate nu cunosc așa bine cu ce se mănâncă politica dar in legătură cu dosarul domnului Georgescu, vineri seară cand am văzut acel protest la poarta Cotroceniului mi-am pus întrebarea, ce se va mai întâmpla in următoarele zile,eu am impresia că se aflase "pe surse" de rezultatul acelui dosar si de aia au ieșit. Iar în ceea ce-l privește pe Fritz părerea mea este că a vrut să arate încă o dată că ei conduc România...O zi bună tuturor.

Rezultat 3
Scor: 0.57
Text: Cite abuzuri , cită lăcomie, cită dictatura mascată in România ,COVID-19 oameni aruncați în saci că gunoiul și Biserica Ortodoxă nicăieri...are o mare dreptate d- na Diana Sosoaca ...nu toți popi sunt pe calea Larga dar sunt destui și că o putere în poporul acesta nu ia tras la răspundere

### Observatie
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- Din cele 5 rezultate, aproximativ 3 sunt relevante clar pentru bula „Personalist-salvator”.
- Textele recuperate exprimă în mare parte vocea agentului: susținere pentru lider, neîncredere în sistem și speranța că acesta poate salva țara.
- Rezultatul 4 este cel mai potrivit, pentru că vorbește direct despre lider ca salvator și despre eliminarea „hoților din parlament”.
- Rezultatele 1, 2 și 5 sunt parțial relevante, dar sunt mai ambigue sau mai greu de interpretat.
- Rezultatul 3 pare mai slab pentru această bulă, deoarece este mai mult anti-sistem/conspiraționist decât personalist-salvator.